In [ ]:
from pathlib import Path

import pandas as pd
import polars as pl

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(200)

# Cadrage et segmentation

Ce premier notebook pose le cadre de l'étude et isole notre population cible avant de la segmenter pour modéliser la PD.

**Étapes suivies :**
- **1. Cadrage :** exploration de la base, représentativité, temporalité de la cible, critères d'exclusion et traitement des dossiers joints.
- **2. Segmentation :** sélection univariée des variables discriminantes, segmentation par arbre CART, validation de l'hétérogénéité des risques (analyse d'IV) et arbitrage.
- **3. Analyse descriptive :** comparaison des segments retenus.


In [ ]:
ID, TARGET, DATE = "gdt", "ddefaut_ndb", "datdelhis"
SOURCE = Path("data/_01_99__ensai_par_final.sas7bdat.gz")
CACHE = Path("data/processed/base.parquet")

Chargement : le fichier SAS est converti une seule fois en parquet (lecture plus rapide ensuite). Les `NaN` issus de SAS sont convertis en `null` pour être comptés comme manquants, et `datdelhis` (YYYYMM) est converti en date.

In [ ]:
from utils.exploration import nan_to_null

if not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    pl.from_pandas(pd.read_sas(SOURCE, encoding="latin1")).write_parquet(CACHE)

# gestion des valeurs manquantes
base = nan_to_null(pl.read_parquet(CACHE)).with_columns(
    pl.date(pl.col(DATE).cast(pl.Int32) // 100, pl.col(DATE).cast(pl.Int32) % 100, 1).alias(DATE)
)
base.shape

# Partie 1 — Cadrage de la population
## 1.1 Structure de la base

In [ ]:
print("lignes :", base.height)
print(f"{ID} distincts :", base[ID].n_unique())
print(f"{ID} x {DATE} distincts :", base.select(ID, DATE).n_unique())
print("dates d'observation :", base[DATE].unique().sort().to_list())
print(base.group_by(ID).agg(nb_mois=pl.len()).group_by("nb_mois").agg(nb_clients=pl.len()).sort("nb_mois"))

- `gdt × datdelhis` est unique : une ligne = un dossier observé à une date.
- Seulement **4 mois d'observation** (janvier à avril 2021) :
  - la stabilité temporelle se mesure sur ces 4 mois ;
  - l'échantillon Out-of-Time sera **202104** (avril 2021) ;
  - les 4 mois, avec leur fenêtre de défaut à 12 mois, couvrent une **seule année de défauts** (2021, période Covid atypique). La LRA, moyenne sur un cycle complet (≥ 5 ans), n'est donc pas estimable sur la base : au calibrage, on retiendra une approximation avec une MoC de catégorie A.
- La grande majorité des dossiers n'apparaît qu'**un seul mois** : c'est un échantillon mensuel, pas un panel. Le découpage apprentissage / test se fera néanmoins **par `gdt`**, pour qu'aucun dossier ne soit des deux côtés.


## 1.2 Plan de sondage

Le défaut étant un événement rare, la base est issue d'un **sous-échantillonnage stratifié** (conservation de tous les défauts et d'une fraction des clients sains) :

* **`SelectionProb` ($\pi_i$)** : Probabilité qu'avait l'observation d'être retenue dans la base ($\approx 1$ pour les défauts, $< 1$ pour les sains).
* **`SamplingWeight` ($w_i$)** : Poids de sondage ($w_i = 1 / \pi_i$). Indique le nombre de clients réels du portefeuille représentés par cette ligne.

**Impacts clés sur le projet :**
1. **Taux de défaut brut** : Artificiellement gonflé dans la base ; calculer la moyenne pondérée par `SamplingWeight` pour retrouver le taux réel.
2. **Modélisation (Gini / Scorecard)** : L'ordonnancement et les coefficients $\beta$ ne sont pas biaisés par ce plan de sondage.
3. **Calibrage (PD réelle)** : Nécessite de redresser les probabilités ou de corriger l'intercept ($\beta_0$).

 On va donc vérifier si ce tirage a déformé la proportion de défauts.


In [ ]:
print(base.group_by(TARGET).agg(
    n=pl.len(),
    n_poids_distincts=pl.col("SamplingWeight").n_unique(),
    poids_min=pl.col("SamplingWeight").min(),
    poids_max=pl.col("SamplingWeight").max(),
))
print("taux brut    :", base[TARGET].mean())
print("taux pondéré :", base.select((pl.col("SamplingWeight") * pl.col(TARGET)).sum() / pl.col("SamplingWeight").sum()).item())

Les poids de sondage valent tous environ 66,67, que ce soit pour les sains ou pour les défauts. Ça correspond à un tirage d'environ 1,5 % de la population (1 / 66,67). On trouve 4 valeurs de poids différentes par classe, ce qui fait 8 strates : le tirage a vraisemblablement été stratifié par mois et par statut (sain / défaut), avec le même taux de tirage dans chaque strate (allocation proportionnelle). Les défauts n'ont donc pas été sur-représentés, contrairement à ce qu'on fait souvent quand la cible est rare.

**Conséquence pour le calibrage**

Le taux de défaut brut (0,781 %) et le taux pondéré (0,781 %) sont quasiment identiques : l'échantillon est représentatif du portefeuille sur la période observée. On n'aura donc pas besoin de corriger l'intercept au moment du calibrage (ch. 1.3).

**Exclusion de `SelectionProb` et `SamplingWeight`**

Les poids des défauts (entre 66,564 et 66,657) et ceux des sains (entre 66,666 et 66,667) ne se chevauchent pas : connaître le poids d'une ligne suffit à savoir si c'est un défaut. Ces deux variables contiennent donc directement l'information de la cible (fuite). On les exclut de toute la suite : segmentation, sélection de variables et modèles.


## 1.3 La cible dans le temps
`ddefaut_ndb` = passage en défaut dans les 12 mois suivant la date d'observation.

In [ ]:
from utils.exploration import risk_rate_by_modality, chi2_association

print(risk_rate_by_modality(base, DATE, TARGET).sort(DATE))

avril = pl.col(DATE) == pl.date(2021, 4, 1)
print(chi2_association(base.with_columns(avril=avril), "avril", TARGET))


Les volumes sont stables d'un mois à l'autre (environ 78 000 lignes, soit un quart de la base chaque mois). Le taux de défaut est stable de janvier à mars (autour de 0,75 %), puis monte à 0,87 % en avril. Avec le taux de janvier-mars, on s'attendait à environ 588 défauts en avril, on en observe 679 : l'écart est significatif (test du Chi², p == 0.0015074549 < 0,05).

- Si la fenêtre de 12 mois était incomplète pour les derniers mois, le taux baisserait en fin de période. Ici il augmente, donc la cible semble bien complète.
- La hausse d'avril vient surtout des dossiers sans engagement.
- À garder en tête : l'échantillon OOT (avril) aura un taux de défaut plus élevé que l'échantillon d'apprentissage. Ça ne devrait pas beaucoup jouer sur le Gini, qui mesure l'ordonnancement et pas le niveau, mais il faudra en tenir compte au calibrage.


## 1.4 Définition du périmètre
### Variables de périmètre constantes

In [ ]:
for v in ["type_PM_PP", "code_marche_v2"]:
    print(risk_rate_by_modality(base, v, TARGET))

Les deux variables n'ont qu'une seule modalité, sans valeur manquante : tous les dossiers sont des personnes physiques (PP) du marché des particuliers. Il n'y a aucun filtre à ajouter. Comme ces deux variables sont constantes, elles n'apportent aucune information : on les retire de la modélisation.

### Dossiers déjà en défaut à la date d'observation
Un dossier déjà en défaut au moment de l'observation n'a pas de probabilité de défaut à estimer, puisque le défaut a déjà eu lieu. Si on le garde, le modèle apprend à reconnaître des défauts existants au lieu de les prédire, ce qui gonfle artificiellement le Gini. On regarde donc quelles variables permettent de repérer ces dossiers.


In [ ]:
from utils.exploration import controle_groupes

print(base.select(
    pl.col("DATE_DERNIER_DFO").min().alias("dernier_dfo_min"),
    pl.col("DATE_DERNIER_DFO").max().alias("dernier_dfo_max"),
    pl.col(DATE).min().alias("observation_min"),
))

volumes, recouvrement = controle_groupes(base, {
    "arr_90j": pl.col("NBJ_ARR_SS") >= 90,
    "dfo_non_resolu": pl.col("DATE_DERNIER_DFO") >= pl.col("DATE_SAIN"),
    "date_sain_null": pl.col("DATE_SAIN").is_null(),
    "dfo_M1_M3": pl.col("Dfo_max_Mm1_Mm3") == 1,
}, TARGET)
print(volumes)
print(recouvrement)

**Constat**
- `NBJ_ARR_SS ≥ 90` (au moins 90 jours d'arriérés significatifs) : c'est la définition réglementaire du défaut. Sur ces 170 dossiers, 72 % sont en défaut dans l'année. Ce n'est pas 100 %, probablement parce que certains régularisent leur situation ou parce que la cible ne compte que les nouvelles entrées en défaut. Ils représentent 5 % des défauts pour seulement 0,05 % de la population : on les **exclut**.
- `Dfo_max_Mm1_Mm3 = 1` (défaut au cours des 3 derniers mois) : 25 % de ces 452 dossiers font défaut dans l'année. C'est beaucoup plus que la moyenne (0,78 %), mais loin de 100 % : ce ne sont pas des défauts en cours, plutôt des dossiers récemment sortis du défaut. Cette information est connue à la date d'observation, donc on les **garde** ; la variable servira de risk driver.
- `DATE_SAIN` manquante (14 lignes) et défaut non résolu (2 lignes) : très peu de lignes et aucun défaut, on les garde.
- `DATE_DERNIER_DFO` ne peut pas servir à repérer les défauts à date : elle s'arrête au 31/12/2020, avant la première observation. D'ailleurs, aucun des 170 dossiers à 90 jours d'arriérés ou plus n'y apparaît comme en défaut. Le seul indicateur fiable est donc `NBJ_ARR_SS`.



### Banque Privée / Gestion de Fortune

In [ ]:
print(risk_rate_by_modality(base, "cod_axe_unite_max", TARGET).sort("cod_axe_unite_max"))

**Constat** : Banque Privée (2) et Gestion de Fortune (3) représentent ≈ 3,7 % des dossiers et ≈ 35 défauts. Ce sont bien des particuliers, et un modèle PD doit couvrir tout le portefeuille → **gardés**. Trop peu de défauts pour un segment propre : ce sera un risk driver (modalités 2 et 3 regroupées).

### Mineurs

In [ ]:
from utils.exploration import controle_groupes

AGE_ANS = pl.coalesce("AGE_PP_EN_MOIS", "AGE_PP_EN_MOIS_med") / 12  # âge des dossiers joints dans _med 

volumes, _ = controle_groupes(base, {"mineurs": AGE_ANS < 18}, TARGET)
print(volumes)

**Constat** : 48 407 dossiers sont des mineurs, soit 15,5 % de la base, pour seulement 24 défauts (0,05 %, seize fois moins que la moyenne). Un mineur n'a pas la capacité juridique de contracter un crédit, donc il n'y a pas vraiment de risque de crédit à modéliser pour lui. Les garder ajouterait beaucoup de dossiers sains très faciles à classer, ce qui gonflerait le Gini sans rien apporter : on les **exclut** du périmètre (ils seraient notés par une règle simple).

L'âge reste par contre une variable candidate pour le modèle. On décidera de l'utiliser ou non lors de la sélection de variables, en comparant le Gini avec et sans l'âge, car c'est un critère sensible du point de vue de la non-discrimination.

### Application des filtres


In [ ]:
from utils.exploration import appliquer_filtres

base_perim, suivi_perimetre = appliquer_filtres(base, {
    "défaut à date (NBJ_ARR_SS >= 90)": pl.col("NBJ_ARR_SS") >= 90,
    "mineurs ": AGE_ANS < 18,
}, TARGET)
print(suivi_perimetre)

## 1.5 Dossiers joints (`nb_tiers > 1`)
Pour les dossiers à plusieurs clients, les variables individuelles (âge, ancienneté, comportement de compte…) ne sont disponibles que sous forme agrégée : `_min`, `_max`, `_med`. On vérifie le remplissage.

In [ ]:
VARS_TEST = ["AGE_PP_EN_MOIS", "ANCIENNETE_modif", "CRTAD_IND_0042", "ENCOURS_PAR"]
print(base_perim.group_by(joint=pl.col("nb_tiers") > 1).agg(
    pl.len().alias("n"),
    *[pl.col(v).is_not_null().mean().alias(f"{v}_rempli") for v in VARS_TEST],
    *[pl.col(f"{v}_med").is_not_null().mean().alias(f"{v}_med_rempli") for v in VARS_TEST],
).sort("joint"))

Pour un dossier qui regroupe plusieurs clients (par exemple un couple), les variables propres à chaque personne (âge, ancienneté, comportement de compte…) ne sont pas renseignées directement. La base donne à la place le minimum, le maximum et la médiane des membres du dossier (`_min`, `_max`, `_med`). Le tableau le confirme : les dossiers individuels ont toujours la variable de base et jamais la version `_med`, et c'est l'inverse pour les dossiers joints. Si on ne fait rien, tous les dossiers joints apparaîtront comme « manquants » sur ces variables, et la classe « manquant » voudra en fait dire « dossier joint ».

**Ce qu'on fait** : on regroupe les deux en une seule colonne, en prenant la valeur individuelle quand elle existe et la médiane du dossier sinon (fonction `coalesce`). Pour un couple, la médiane correspond à la moyenne des deux membres.
- Trois codes n'ont pas de médiane, seulement un min et un max : les deux indicateurs d'interdit bancaire et l'axe de clientèle. Une médiane n'aurait pas de sens pour un code : on prend le max, par prudence. Si un membre du couple est interdit bancaire, on considère que le dossier l'est.
- SAS limite les noms de variables à 32 caractères, donc certaines versions `_med` ont un nom raccourci (ex. `RATIO_SOLDE_MOYEN_SUR_EPAR_med` pour `RATIO_SOLDE_MOYEN_SUR_EPARGNE`). On les associe à la main.


In [ ]:
from utils.exploration import unifier_joints

TRONQUES = {
    "RATIO_SOLDE_MOYEN_SUR_EPAR": "RATIO_SOLDE_MOYEN_SUR_EPARGNE",
    "RATIO_NBJDEB_NONAUTO_SUR_NBJ": "RATIO_NBJDEB_NONAUTO_SUR_NBJDEB",
}
base_perim, rapport_joints = unifier_joints(
    base_perim, TRONQUES, max_seul=("CODITDBDF_PAR", "CODITDBDF_PRO", "cod_axe_unite"))
print(rapport_joints)
print(base_perim.select(pl.col(VARS_TEST).null_count()))

**Limites** :
- Pour les montants (encours, épargne…), la médiane d'un couple correspond à la moyenne des deux membres, donc elle sous-estime ce que possède le foyer : un total serait plus parlant.
- La CSP (`CODACVPRO_modif`) n'a pas de version agrégée : elle reste manquante pour les dossiers joints, qui formeront une modalité à part.
- L'impact reste limité, puisque les dossiers joints représentent environ 12 % des défauts.

**Piste pour la suite (feature engineering)** : construire des variables propres aux dossiers joints à partir des `_min`, `_max` et `_med`, par exemple le total du foyer (médiane × nombre de membres, exact pour un couple) ou l'écart entre les membres (max − min), qui peut révéler un membre en difficulté.


# Partie 2 — Segmentation

On cherche à découper la population en quelques groupes (2 à 4) qui auront chacun leur propre modèle. La démarche :
- **2.1** les critères de choix ;
- **2.2** les variables candidates, testées une par une ;
- **2.3** les croisements proposés par un arbre de décision ;
- **2.4** les découpages candidats : volumes et stabilité dans le temps ;
- **2.5** le test formel du cours (modèle poolé contre modèle avec interactions), puis la recherche d'un meilleur découpage ;
- **2.6** la décision.

## 2.1 Critères de choix

Une variable très liée au défaut n'est pas forcément une bonne variable de segmentation. On juge chaque découpage sur quatre critères :

1. **Sens métier** : les segments doivent correspondre à des clientèles qu'on sait décrire.
2. **Des risk drivers différents** : on appelle risk drivers les variables qui prédisent le défaut dans un groupe. Si ce sont les mêmes variables, avec le même effet, dans chaque segment, un seul modèle qui contient la variable de segmentation fait aussi bien que plusieurs modèles séparés. Un simple écart de taux de défaut ne suffit donc pas.
3. **Assez de défauts par segment** (au moins quelques centaines) pour estimer un modèle.
4. **Stabilité dans le temps** : les écarts entre segments doivent se retrouver tous les mois.

Le critère 2 est tranché par le **test formel du cours** (partie 2.5) : on compare un modèle poolé, où les variables ont le même effet dans tous les segments, à un modèle avec interactions, où chaque segment a ses propres coefficients.

Dans une première version, on avait comparé les classements d'IV des segments. On a abandonné cette approche : elle regarde les variables une par une, alors que le test les regarde ensemble dans un modèle, et les deux méthodes ne donnaient pas les mêmes conclusions (notamment pour le type de crédit).

### Variables construites pour la segmentation
Pour tester des segmentations, on construit à partir des variables brutes quelques indicateurs simples, qui décrivent la relation du client avec la banque.

| Variable | Construction | Ce qu'elle représente |
|---|---|---|
| `type_credit` | règle par ordre de priorité : crédit immobilier, sinon crédit conso (perso, flex ou revolving), sinon crédit MLT, sinon autre engagement, sinon aucun crédit | le principal type de crédit détenu (un client qui a plusieurs crédits est classé selon le premier de la liste) |
| `has_revolving` | au moins un crédit revolving | détention d'un crédit renouvelable |
| `has_decouvert` | encours sur le compte courant > 0 | utilisation d'un découvert autorisé |
| `joint` | `nb_tiers > 1` | dossier à plusieurs titulaires (multi-détenu) |
| `anciennete_mois`, `anciennete` | ancienneté de la relation, en mois puis en 4 tranches (< 1 an, 1-2 ans, 2-5 ans, 5 ans et plus) | clients récents ou anciens |
| `age_ans`, `age` | âge en années, puis en 4 tranches (18-25, 25-40, 40-60, 60 ans et plus) | âge du client (médiane du foyer pour les dossiers joints) |
| `compte_inactif` | aucune opération dans le mois | compte qui n'est pas utilisé |
| `historique_defaut` | date de dernier défaut renseignée | le client a déjà été en défaut par le passé |
| `flux_pro` | indicateur de mouvements d'affaires | client qui a une activité professionnelle |
| `patrimonial` | axe de clientèle Banque Privée ou Gestion de Fortune | clientèle patrimoniale |

On fait une hypothèse : quand un nombre de contrats ou un encours est manquant, on considère que le client n'a pas ce produit (valeur 0).

La fonction crée aussi deux des découpages testés plus bas (par type de crédit et par ancienneté) ; les autres sont construits en 2.4.

In [ ]:
from utils.segmentation import construire_candidats

cand = construire_candidats(base_perim)
print(cand.group_by("type_credit").agg(n=pl.len(), n_defaut=pl.col(TARGET).sum(), taux=pl.col(TARGET).mean()).sort("type_credit"))


## 2.2 Candidats testés un par un

In [ ]:
from utils.segmentation import rank_segmentation_candidates

CANDIDATS = ["has_engagement", "type_credit", "has_revolving", "has_decouvert", "joint", "anciennete",
             "compte_inactif", "historique_defaut", "flux_pro", "patrimonial", "age", "CODACVPRO_modif"]
print(rank_segmentation_candidates(cand, CANDIDATS, TARGET))

In [ ]:
from utils.exploration import risk_rate_by_modality

for v in CANDIDATS:
    print(risk_rate_by_modality(cand, v, TARGET).sort(v))

Le tableau classe les candidats selon leur lien avec le défaut (V de Cramer). Ce classement ne suffit pas pour choisir : une variable très liée au défaut est souvent un bon risk driver, qu'il vaut mieux garder comme variable du modèle plutôt que de couper la population dessus. On regarde donc aussi le nombre de défauts par modalité et le sens métier.

Candidats écartés :

| Candidat | Pourquoi on l'écarte |
|---|---|
| `historique_defaut` | c'est la variable la plus liée au défaut (6 % de défaut contre 0,75 % pour les autres), mais elle ne concerne que 2,4 % des dossiers et 374 défauts. C'est un risk driver, qu'on gardera dans le modèle |
| `patrimonial` | seulement 31 défauts (critère 3) |
| `flux_pro` | manquant pour presque 99 % des dossiers |
| `joint` | seulement 284 défauts pour un segment à lui seul ; la séparation mono / multi-détenus recommandée par le référent est testée en 2.4 et 2.5 |
| `CODACVPRO_modif` (CSP) | 59 modalités, dont plusieurs sans aucun défaut : il faudra les regrouper au binning, ça ne peut pas servir de coupure |
| `has_revolving` | 214 défauts et 6 % des dossiers seulement (critère 3) |
| `has_decouvert` | même lien avec le défaut que `has_engagement`, et les deux se recoupent beaucoup (le découvert est un type d'engagement). On préfère `has_engagement`, qui couvre toutes les formes de crédit et se comprend mieux côté métier |
| `anciennete`, `age` | le risque baisse nettement avec l'ancienneté (de 3 % pour moins d'un an à 0,7 % au-delà de 5 ans) ; pour l'âge, il est le plus élevé entre 25 et 40 ans (1,2 %) puis baisse jusqu'à 0,4 % après 60 ans. Ce sont surtout des risk drivers ; l'ancienneté, proposée par les arbres (2.3), est quand même testée comme découpage en 2.4 et 2.5 |
| `compte_inactif` | aucun lien avec le défaut (0,88 % contre 0,87 %, p = 0,95) |
| crédit immobilier seul | 121 défauts (critère 3) |


**Candidats retenus.** Prises une par une, ce sont les deux variables qui présentent le plus fort potentiel de segmentation : elles ont un sens métier clair, assez de défauts dans chaque groupe et un lien réel avec le défaut.

| Candidat | Pourquoi on le garde |
|---|---|
| `has_engagement` | sépare les clients sans aucun crédit (515 défauts, 0,55 %) de ceux qui ont au moins un engagement (1 779 défauts, 1,05 %) : deux relations avec la banque très différentes, et assez de défauts des deux côtés (critères 1 et 3). Son lien avec le défaut est modéré (V = 0,026) |
| `type_credit` | précise le type d'engagement : aucun crédit (515 défauts), facilités seules comme le découvert ou la carte à débit différé (1 153), crédit amortissable immobilier, conso ou MLT (626 après regroupement). C'est le candidat le plus lié au défaut parmi ceux qui ont un sens de segmentation (V = 0,040). La modalité « MLT » est négligeable (107 dossiers, 2 défauts) et sera regroupée avec les autres crédits amortissables |






## 2.3 Croisements de variables par arbre CART
Les tests univariés ne voient pas les croisements. Un arbre peu profond (profondeur 2 à 3, feuilles d'au moins 5 % de la population) propose des découpages multivariés ; chaque feuille est un segment candidat. On le lance sur toute la population, puis sur les seuls mono-détenus.

In [ ]:
from utils.segmentation import tree_segmentation

STRUCTURELLES = ["has_engagement", "type_credit", "has_decouvert", "has_revolving", "joint",
                 "compte_inactif", "patrimonial", "anciennete_mois"]

for prof in (2, 3):
    feuilles, _ = tree_segmentation(cand, STRUCTURELLES, TARGET, max_depth=prof, min_leaf_share=0.05)
    print(f"=== toute la population | profondeur {prof}")
    print(feuilles)


In [ ]:
mono = cand.filter(~pl.col("joint"))
STRUCT_MONO = [v for v in STRUCTURELLES if v != "joint"]
for prof in (2, 3):
    feuilles, _ = tree_segmentation(mono, STRUCT_MONO, TARGET, max_depth=prof, min_leaf_share=0.05)
    print(f"=== mono-détenus | profondeur {prof}")
    print(feuilles)

**Constat** : l'arbre cherche les coupures qui font le plus varier le taux de défaut, sans dire si les risk drivers changent d'un groupe à l'autre.

- Sur toute la population, la première coupure est l'**ancienneté** (environ 13,5 ans), puis le **type de crédit**. Parmi les clients récents, ceux qui **n'ont que des facilités** (découvert, carte à débit différé) sont les plus risqués (2,3 %, et même 3,7 % pour ceux arrivés depuis moins de 5 ans) ; parmi les anciens, ce sont ceux qui ont un crédit conso. À l'opposé, les clients de plus de 32 ans sans crédit conso ne font que 0,2 % de défaut.
- Sur les seuls mono-détenus, l'arbre coupe d'abord à 4 ans d'ancienneté, puis sur le découvert et le crédit conso.
- Ajouter l'âge et l'historique de défaut ne change pas les arbres (l'historique de défaut concerne de toute façon moins de 5 % des dossiers, la taille minimale des feuilles).

Deux pistes en ressortent pour la suite : le **type de crédit**, et en particulier le groupe des clients qui n'ont que des facilités, qui confirme le candidat retenu en 2.2 ; et l'**ancienneté**, qui n'était pas retenue variable par variable mais qu'on testera quand même comme découpage (seuils de 5 ans et de 4 ans) en 2.4 et 2.5.


### 2.3 bis — Structure de la population (analyse non supervisée)

Les arbres cherchent les coupures qui font varier le taux de défaut. On complète par une analyse qui n'utilise pas la cible : comment la population se structure-t-elle d'elle-même ?

- une **AFDM** (analyse factorielle de données mixtes, l'équivalent de l'ACP quand on mélange variables quantitatives et qualitatives) sur les variables qui décrivent la relation du client avec la banque. Si une variable pèse fortement sur les premiers axes, elle structure réellement la population et mérite d'être testée comme critère de segmentation ;
- une **classification k-means** sur les premiers axes, pour voir si des groupes homogènes apparaissent, et s'ils ressemblent aux découpages candidats.

Un groupe qu'on ne sait pas décrire par une règle simple au moment de l'octroi n'est pas utilisable comme segment, aussi net soit-il statistiquement : on lit donc chaque groupe côté métier. L'âge est inclus pour décrire la population, mais il ne pourrait pas servir de critère de segmentation (critère sensible).


In [ ]:
from utils.segmentation import afdm, clusters_kmeans, profil_groupes

QUANTI_AFDM = ["anciennete_mois", "age_ans", "nb_produits_distincts"]
QUALI_AFDM = ["type_credit", "has_decouvert", "has_revolving", "joint", "compte_inactif", "patrimonial"]

resultat_afdm = afdm(cand, QUANTI_AFDM, QUALI_AFDM, n_axes=5)
print(resultat_afdm["variance"])
print(resultat_afdm["liens"].with_columns(pl.col("^axe_.*$").round(2)))


In [ ]:
import matplotlib.pyplot as plt

COULEURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
ech = cand.with_columns(resultat_afdm["coordonnees"]).with_columns(
    seg_D4=pl.when(pl.col("joint")).then(pl.lit("multi"))
    .when(pl.col("has_engagement") == 0).then(pl.lit("mono sans engagement"))
    .otherwise(pl.lit("mono avec engagement"))
).sample(5_000, seed=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor="#fcfcfb")
for ax, (col, titre) in zip(axes, [("seg_type_credit", "D2 type de crédit"), ("seg_D4", "D4 mono/multi + engagement")]):
    for i, seg in enumerate(sorted(ech[col].unique().to_list())):
        d = ech.filter(pl.col(col) == seg)
        ax.scatter(d["axe_1"], d["axe_2"], s=8, alpha=0.5, color=COULEURS[i], label=seg, edgecolors="none")
    ax.set_title(f"Plan factoriel (axes 1 et 2), coloré selon {titre}", loc="left", fontsize=11, color="#0b0b0b")
    ax.set_xlabel("axe 1", color="#52514e"); ax.set_ylabel("axe 2", color="#52514e")
    ax.set_facecolor("#fcfcfb")
    for cote in ("top", "right"):
        ax.spines[cote].set_visible(False)
    ax.tick_params(colors="#52514e", length=0)
    ax.legend(fontsize=9, frameon=False, markerscale=2, labelcolor="#52514e")
fig.tight_layout()
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
fig.savefig("outputs/figures/afdm_plan_factoriel.png", dpi=150, facecolor="#fcfcfb")
plt.show()


In [ ]:
from utils.exploration import chi2_association

qualite, labels = clusters_kmeans(resultat_afdm["coordonnees"], k_values=(2, 3, 4), n_axes=4)
print(qualite)

K = int(qualite.sort("silhouette", descending=True)["k"][0])   # nombre de groupes le mieux séparé
cand = cand.with_columns(cluster=pl.Series([f"C{c + 1}" for c in labels[K]]))
print(f"k retenu : {K}")

for col in ["seg_type_credit", "seg_mono_multi" if "seg_mono_multi" in cand.columns else "has_engagement"]:
    v = chi2_association(cand, "cluster", col)["cramers_v"]
    print(f"--- groupes k-means × {col} (V de Cramer = {v:.2f})")
    print(cand.group_by("cluster", col).len().pivot(on=col, index="cluster", values="len").sort("cluster"))

print(profil_groupes(cand, "cluster", TARGET, QUANTI_AFDM, QUALI_AFDM))


**Constat** : l'AFDM fait ressortir deux grandes dimensions de la population. Le premier axe (21 % de la variance) est porté par l'équipement en crédit : type de crédit, nombre de produits, découvert et revolving. Le deuxième (15 %) correspond au cycle de vie du client : ancienneté et âge. Le nombre de titulaires (multi-détenus) ne pèse presque pas sur les premiers axes. Sur le plan factoriel, les trois segments du type de crédit (D2) occupent des zones bien distinctes, alors que les multi-détenus sont mélangés aux autres.

Le k-means sur les quatre premiers axes donne la meilleure séparation avec 4 groupes (silhouette de 0,47, contre 0,41 pour 3 et 0,32 pour 2). Ces groupes sont faciles à décrire :
- C2 : clients sans aucun crédit (94 339 dossiers, 0,55 % de défaut), exactement le segment S1 ;
- C3 : clients qui n'ont que des facilités (104 543 dossiers, 1,10 %), exactement le segment S2 ;
- C1 : clients avec un crédit immobilier, souvent en couple (30 431 dossiers, 0,38 %) ;
- C4 : clients avec un crédit conso, qui utilisent souvent leur découvert (34 377 dossiers, 1,46 %).

Sans utiliser la cible, la population se structure donc presque exactement selon le type de crédit (V de Cramer ≈ 1 avec D2), et beaucoup moins selon la séparation mono / multi-détenus (V = 0,68 avec D4). C'est un argument indépendant en faveur de D2. Le k-means coupe en plus le segment « crédit amortissable » entre immobilier et conso, qui ont des niveaux de risque très différents (0,38 % contre 1,46 %). Mais le crédit immobilier seul ne compte qu'environ 120 défauts, trop peu pour un modèle (critère 3) : cet écart sera pris en compte par la variable type de crédit à l'intérieur du segment.


## 2.4 Découpages candidats

On construit les découpages à tester à partir de trois sources : les candidats retenus variable par variable en 2.2 (présence d'un engagement, type de crédit), les coupures proposées par les arbres en 2.3 (ancienneté), et la recommandation de notre référent de séparer les dossiers mono-détenus et multi-détenus. Cette dernière se défend côté métier : un foyer n'est pas un individu, et les informations disponibles ne sont pas les mêmes pour les dossiers joints (médianes du foyer, voir 1.5). Les multi-détenus ne comptent que 284 défauts : ils forment donc toujours un seul segment, et c'est le groupe mono-détenus qu'on coupe.

| | Découpage | Segments | Origine |
|---|---|---|---|
| D1 | sans / avec engagement | 2 | 2.2 |
| D2 | type de crédit : sans engagement / facilités seules (découvert, carte) / crédit amortissable (immo, conso, MLT) | 3 | 2.2, confirmé par les arbres |
| D3 | ancienneté de la relation, dans le groupe avec engagement : sans engagement / avec engagement et client depuis ≤ 5 ans / depuis > 5 ans | 3 | arbres (2.3) |
| D4 | mono / multi, puis engagement : multi / mono sans engagement / mono avec engagement | 3 | référent + 2.2 |
| D5 | mono / multi, puis type de crédit : multi / mono sans engagement / mono facilités / mono crédit amortissable | 4 | référent + 2.2 |
| D6 | mono / multi, puis ancienneté : multi / mono ≤ 4 ans / mono > 4 ans | 3 | référent + arbre sur les mono-détenus |
| D7 | mono / multi, puis crédit amortissable : multi / mono avec crédit amortissable / mono sans | 3 | référent + 2.2 |

Ces sept découpages sont fixés avant de regarder les résultats des tests de la partie 2.5.


In [ ]:
from utils.segmentation import risk_rate_over_time

multi = pl.col("nb_tiers") > 1
engage = pl.col("has_engagement") == 1

cand = cand.with_columns(
    seg_engagement=pl.when(engage).then(pl.lit("SB_avec_engagement")).otherwise(pl.lit("SA_sans_engagement")),
    seg_mono_multi=pl.when(multi).then(pl.lit("SM_multi"))
    .when(~engage).then(pl.lit("SA_mono_sans_eng"))
    .otherwise(pl.lit("SB_mono_avec_eng")),
    seg_mono_type=pl.when(multi).then(pl.lit("SM_multi"))
    .when(pl.col("type_credit") == "5_sans_credit").then(pl.lit("SA_mono_sans_eng"))
    .when(pl.col("type_credit") == "4_autre_engagement").then(pl.lit("SF_mono_facilites"))
    .otherwise(pl.lit("SC_mono_amort")),
    seg_mono_anciennete=pl.when(multi).then(pl.lit("SM_multi"))
    .when(pl.col("anciennete_mois") <= 48).then(pl.lit("SR_mono_recent"))
    .otherwise(pl.lit("SN_mono_ancien")),
    seg_mono_amort=pl.when(multi).then(pl.lit("SM_multi"))
    .when(pl.col("type_credit").is_in(["1_immo", "2_conso", "3_MLT"])).then(pl.lit("SC_mono_amort"))
    .otherwise(pl.lit("SF_mono_non_amort")),
)

DECOUPAGES = {
    "D1 sans / avec engagement": "seg_engagement",
    "D2 type de crédit": "seg_type_credit",
    "D3 ancienneté (engagés)": "seg_anciennete",
    "D4 mono/multi + engagement": "seg_mono_multi",
    "D5 mono/multi + type de crédit": "seg_mono_type",
    "D6 mono/multi + ancienneté": "seg_mono_anciennete",
    "D7 mono/multi + crédit amortissable": "seg_mono_amort",
}



volumes_decoupages = pl.concat([
    cand.group_by(col).agg(n=pl.len(), n_defaut=pl.col(TARGET).sum(), taux=pl.col(TARGET).mean())
    .rename({col: "segment"}).with_columns(decoupage=pl.lit(nom))
    for nom, col in DECOUPAGES.items()
]).select("decoupage", "segment", "n", "n_defaut", "taux").sort("decoupage", "segment")
print(volumes_decoupages)

**Constat — volumes**

| | Découpage | Plus petit segment |
|---|---|---|
| D1 | sans / avec engagement | 515 défauts (sans engagement) |
| D2 | type de crédit | 515 défauts (sans engagement) |
| D3 | ancienneté (engagés) | 515 défauts (sans engagement) |
| D4 | mono/multi + engagement | 284 défauts (multi-détenus) |
| D5 | mono/multi + type de crédit | 284 défauts (multi-détenus) |
| D6 | mono/multi + ancienneté | 284 défauts (multi-détenus) |
| D7 | mono/multi + crédit amortissable | 284 défauts (multi-détenus) |

Tous les segments ont au moins 284 défauts : les sept découpages respectent le critère 3. Les multi-détenus sont le segment le plus petit, ce qui limiterait la complexité de leur modèle.

Certains découpages séparent des groupes qui ont presque le même taux de défaut (par exemple facilités seules et crédit amortissable, autour de 1 %). Ce n'est pas un problème en soi : une segmentation se justifie par des risk drivers différents, pas par des niveaux différents, puisqu'un écart de niveau peut être pris en compte par une simple variable dans un modèle unique. C'est le test de la partie 2.5 qui tranche.


## 2.5 Test formel : modèle poolé contre modèle avec interactions

C'est le test proposé dans le cours. Pour chaque découpage :
- **modèle poolé** : une régression logistique sur toute la population, avec les mêmes coefficients pour tous les segments (seule la constante change d'un segment à l'autre) ;
- **modèle avec interactions** : chaque variable reçoit un coefficient propre à chaque segment (variable × indicatrice du segment), ce qui revient à estimer un modèle par segment.

On compare les deux avec :
- le **test du rapport de vraisemblance (LR)** : s'il est significatif, les coefficients diffèrent entre segments. Avec 263 690 dossiers, il est significatif dès que les différences sont un peu marquées, donc on le complète par :
- l'**AIC** et le **BIC**, qui pénalisent chaque coefficient ajouté (de 2 pour l'AIC, d'environ 12,5 pour le BIC, qui est donc plus sévère). Plus le critère est bas, meilleur est le modèle.




In [ ]:
from utils.binning import iv_ranking
from utils.segmentation import variables_non_redondantes, test_interactions_segment

EXCLURE = {ID, TARGET, DATE, "SelectionProb", "SamplingWeight", "type_PM_PP", "code_marche_v2","CODACVPRO_modif", "AGE_PP_EN_MOIS"}
VARS = [c for c, t in base_perim.schema.items() if c not in EXCLURE and not t.is_temporal()]
iv_glob = iv_ranking(cand, VARS, TARGET)
VARS_SEG = iv_glob.filter(pl.col("n_classes") <= 15)["variable"].to_list()  # qualitatives à > 15 modalités écartées (IV gonflée)

SEUIL_IV = 0.1  # pouvoir au moins « moyen » selon Siddiqi
candidats = iv_glob.filter(pl.col("variable").is_in(VARS_SEG) & (pl.col("iv") >= SEUIL_IV))
VARS_TEST = variables_non_redondantes(cand, candidats, n=candidats.height)  # toutes celles qui ne sont pas redondantes
print(f"{candidats.height} variables avec une IV ≥ {SEUIL_IV}, dont {len(VARS_TEST)} non redondantes :", VARS_TEST)


In [ ]:
lignes = []
for nom, col in {"D0 aucune (modèle unique)": "seg_unique", **DECOUPAGES}.items():
    donnees = cand.with_columns(seg_unique=pl.lit("tous"))
    g, _ = test_interactions_segment(donnees, VARS_TEST, TARGET, col)
    vol = donnees.group_by(col).agg(n_defaut=pl.col(TARGET).sum())
    lignes.append({
        "decoupage": nom, "n_segments": vol.height, "min_defauts": int(vol["n_defaut"].min()),
        "LR": g["LR"], "ddl": g["ddl"], "pvalue": g["pvalue"],
        "AIC": g["AIC_interactions"], "BIC": g["BIC_interactions"],
    })

comparaison = (
    pl.DataFrame(lignes)
    .with_columns(ecart_AIC=pl.col("AIC") - pl.col("AIC").min(), ecart_BIC=pl.col("BIC") - pl.col("BIC").min())
    .sort("AIC")
    .with_row_index("rang_AIC", offset=1)
)

print(comparaison.select(
    pl.col("rang_AIC").alias("rang"),
    "decoupage",
    pl.col("n_segments").alias("segments"),
    pl.col("min_defauts").alias("min défauts"),
    pl.col("LR").round(0),
    "ddl",
    pl.col("ecart_AIC").round(0).alias("écart AIC"),
    pl.col("ecart_BIC").round(0).alias("écart BIC"),
))


**Constat — comparaison des découpages** (43 variables : IV ≥ 0,1 sur la population globale, sans doublons, CSP et âge retirés ; AIC et BIC du modèle avec un jeu de coefficients par segment, écarts mesurés par rapport au meilleur)

| Rang (AIC) | Découpage | Segments | LR (ddl) | Écart d'AIC | Écart de BIC |
|---|---|---|---|---|---|
| 1 | **D2 type de crédit** | 3 | 517 (81) | **0** | +447 |
| 2 | D5 mono/multi + type de crédit | 4 | 575 (124) | +45 | +953 |
| 3 | D1 sans / avec engagement | 2 | 274 (38) | +227 | +213 |
| 4 | D3 ancienneté (engagés) | 3 | 354 (81) | +234 | +682 |
| 5 | D4 mono/multi + engagement | 3 | 353 (81) | +236 | +683 |
| 6 | D7 mono/multi + crédit amortissable | 3 | 265 (86) | +282 | +782 |
| 7 | D0 aucune (modèle unique) | 1 | — | +423 | **0** |
| 8 | D6 mono/multi + ancienneté | 3 | 162 (86) | +436 | +936 |

- Le test LR est significatif pour tous les découpages (p < 10⁻⁵) : avec autant de dossiers, les coefficients diffèrent toujours un peu d'un segment à l'autre. Ce sont l'AIC et le BIC qui permettent de départager.
- **Selon l'AIC, le type de crédit (D2) est le meilleur découpage.** Il fait nettement mieux que le modèle unique (423 points d'écart) et que tous les autres découpages, sauf D5 qui le suit de près.
- Ajouter la séparation mono / multi-détenus au type de crédit (D5) n'apporte rien : l'AIC se dégrade de 45 points alors qu'on ajoute 43 coefficients. Les multi-détenus n'ont pas assez de spécificités pour justifier leur propre modèle : le nombre de titulaires sera plutôt une variable du modèle. La recommandation du référent seule (D4) arrive loin derrière D2 (+236).
- Les découpages sur l'ancienneté (D3, D6) sont nettement moins bons ; D6 fait même moins bien que le modèle unique. L'ancienneté change surtout le niveau de risque, ce qu'une variable du modèle peut capter.
- **Selon le BIC, aucune segmentation ne fait mieux que le modèle unique.** Avec 43 variables, segmenter en trois ajoute environ 80 coefficients, et le BIC pénalise chacun d'environ 12,5 points : le gain d'ajustement ne compense pas. Avec moins de variables (13), le type de crédit battait le modèle unique au BIC : le verdict du BIC dépend donc surtout du nombre de variables mises dans le test.

Le classement des segmentations est le même quelle que soit la façon de choisir les variables (13 variables globales, les 6 meilleures de chaque groupe, ou toutes celles dont l'IV dépasse 0,1) : **parmi les segmentations, le type de crédit est le meilleur choix.** En revanche, les critères calculés sur l'échantillon d'apprentissage ne permettent pas de trancher entre segmenter et garder un modèle unique : ce sera fait au chapitre 5, en comparant leur Gini hors échantillon.


In [ ]:
from itertools import combinations


def tests_par_paire(col):
    """Test poolé / interactions pour chaque paire de segments d'un découpage."""
    lignes = []
    for a, b in combinations(sorted(cand[col].unique().to_list()), 2):
        g, _ = test_interactions_segment(cand.filter(pl.col(col).is_in([a, b])), VARS_TEST, TARGET, col)
        lignes.append({"paire": f"{a} vs {b}", "LR": round(g["LR"]), "ddl": g["ddl"], "pvalue": g["pvalue"],
                       "AIC_prefere": g["AIC_prefere"], "BIC_prefere": g["BIC_prefere"]})
    return pl.DataFrame(lignes)


for nom, col in [("D2 type de crédit", "seg_type_credit"), ("D4 mono/multi + engagement (référent)", "seg_mono_multi")]:
    print(f"==================== {nom}")
    print(tests_par_paire(col))


**Constat — tests par paire de segments**

| Paire | LR (ddl) | p | L'AIC préfère |
|---|---|---|---|
| sans engagement / facilités seules | 326 (38) | 2·10⁻⁴⁷ | séparer |
| facilités seules / crédit amortissable | 179 (43) | 2·10⁻¹⁸ | séparer |
| sans engagement / crédit amortissable | 142 (38) | 7·10⁻¹⁴ | séparer |
| mono sans engagement / mono avec engagement | 256 (38) | 4·10⁻³⁴ | séparer |
| mono avec engagement / multi-détenus | 118 (43) | 7·10⁻⁹ | séparer |
| mono sans engagement / multi-détenus | 69 (38) | 0,0015 | ne pas séparer |

Dans le type de crédit (D2), les trois segments ont des risk drivers différents deux à deux ; la différence la plus nette oppose les clients sans engagement à ceux qui n'ont que des facilités. Dans le découpage du référent (D4), les multi-détenus se distinguent des mono-détenus avec engagement, mais pas vraiment des mono-détenus sans engagement : l'AIC préfère ne pas les séparer. Le BIC, lui, préfère ne séparer aucune paire, pour la même raison qu'au tableau précédent (le nombre de coefficients ajoutés).

On ne détaille pas ici quelles variables diffèrent : avec 43 variables dont beaucoup sont proches (plusieurs fenêtres d'arriérés ou de soldes), les coefficients se compensent entre eux et ne s'interprètent pas un par un. Seul le test global de chaque paire est fiable.


## 2.6 Décision

On retient **la segmentation par type de crédit (D2)**. La recommandation du référent (D4) est gardée comme alternative, et le modèle unique comme référence : les trois seront comparés au chapitre 5 sur le Gini hors échantillon (apprentissage / test / OOT).

**Segmentation retenue : le type de crédit (D2)**

| Segment | Définition | Dossiers | Défauts | Taux |
|---|---|---|---|---|
| **S1** sans engagement | aucun crédit ni facilité | 94 201 | 515 | 0,55 % |
| **S2** facilités seules | engagement sans crédit amortissable (découvert, carte à débit différé) | 104 788 | 1 153 | 1,10 % |
| **S3** crédit amortissable | au moins un crédit immobilier, conso ou MLT | 64 701 | 626 | 0,97 % |

- **Métier** : trois relations de crédit différentes (pas de crédit, facilités renouvelables, crédit avec échéancier).
- **Test du cours** : c'est le meilleur des sept découpages selon l'AIC, et ce classement ne dépend pas de la façon de choisir les variables du test. Les trois segments ont des risk drivers différents deux à deux.
- **Structure de la population** : l'AFDM et le k-means, qui n'utilisent pas la cible, retrouvent presque exactement ces trois segments (voir 2.3 bis).
- **Volumes** : au moins 515 défauts par segment.

**Alternative : la recommandation du référent, mono / multi-détenus × engagement (D4)**

| Segment | Définition | Dossiers | Défauts | Taux |
|---|---|---|---|---|
| **SM** multi-détenus | `nb_tiers > 1` | 58 947 | 284 | 0,48 % |
| **SA** mono-détenus sans engagement | `nb_tiers = 1` et `has_engagement = 0` | 78 118 | 449 | 0,57 % |
| **SB** mono-détenus avec engagement | `nb_tiers = 1` et `has_engagement = 1` | 126 625 | 1 561 | 1,23 % |

- **Métier** : un foyer n'est pas un individu, et les informations disponibles pour les dossiers joints ne sont pas les mêmes (médianes du foyer).
- **Test du cours** : nettement moins bon que D2 (+236 d'AIC). Les multi-détenus ont des risk drivers proches de ceux des mono-détenus sans engagement (l'AIC préfère ne pas les séparer) et le même niveau de risque. Ajouter la séparation mono / multi au type de crédit (D5) n'améliore pas non plus le modèle.
- **Structure de la population** : les multi-détenus sont répartis dans tous les groupes du k-means, ils ne forment pas un groupe naturel.

Le nombre de titulaires (`nb_tiers`) sera donc utilisé comme variable du modèle plutôt que comme critère de segmentation. Ce point est à discuter avec le référent.

**Points de vigilance**
- **Segmenter ou non.** Selon le BIC, qui pénalise fortement les coefficients ajoutés, aucune segmentation ne fait mieux qu'un modèle unique avec les 43 variables du test ; avec moins de variables, D2 faisait mieux. Les critères calculés sur l'échantillon d'apprentissage ne permettent donc pas de trancher : la comparaison du Gini hors échantillon au chapitre 5 est indispensable.
- **S2 et S3** ont le même niveau de risque (autour de 1 %) : leur séparation repose sur des risk drivers différents, pas sur un écart de niveau.
- **Le taux de défaut de S1 monte en avril** (182 défauts contre 100 à 122 les autres mois), alors qu'avril est l'échantillon OOT : il faudra en tenir compte au calibrage.
- **Sept découpages** ont été comparés sur les mêmes 4 mois : la validation hors échantillon du chapitre 5 confirmera ou non ce choix.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

SEGMENTATION_RETENUE = "seg_type_credit"     # D2 ; alternative : "seg_mono_multi" (D4, recommandation du référent)

COULEURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # palette catégorielle vérifiée pour le daltonisme
MARQUEURS = ["o", "s", "^", "D"]                          # une forme par segment, en plus de la couleur
MOIS = {1: "janv.", 2: "févr.", 3: "mars", 4: "avr."}
Path("outputs/figures").mkdir(parents=True, exist_ok=True)

for titre, col, fichier in [("D2 type de crédit (retenu)", SEGMENTATION_RETENUE, "stabilite_retenue"),
                            ("D4 mono/multi + engagement (alternative)", "seg_mono_multi", "stabilite_alternative")]:
    t = risk_rate_over_time(cand, col, TARGET, DATE)
    x = [MOIS[d.month] for d in t[DATE]]
    fig, ax = plt.subplots(figsize=(10, 4.5), facecolor="#fcfcfb")
    for i, seg in enumerate(sorted(c for c in t.columns if c != DATE)):
        ax.plot(x, t[seg] * 100, color=COULEURS[i], marker=MARQUEURS[i], linewidth=2, markersize=8,
                markeredgecolor="white", markeredgewidth=1.2, label=seg)
    ax.set_title(f"Taux de défaut mensuel par segment — {titre}", loc="left", fontsize=12, color="#0b0b0b")
    ax.set_ylabel("taux de défaut", fontsize=11, color="#52514e")
    ax.set_facecolor("#fcfcfb")
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))
    ax.grid(axis="y", color="#e5e4df", linewidth=0.8)
    ax.set_axisbelow(True)
    for cote in ("top", "right", "left"):
        ax.spines[cote].set_visible(False)
    ax.spines["bottom"].set_color("#b5b4ad")
    ax.tick_params(colors="#52514e", labelsize=11, length=0)
    ax.legend(fontsize=10, frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.5), labelcolor="#52514e")
    fig.tight_layout()
    fig.savefig(f"outputs/figures/{fichier}.png", dpi=150, facecolor="#fcfcfb")
    plt.show()




## Export Excel
On regroupe dans un classeur les tableaux qui justifient les décisions de ce notebook (périmètre et segmentation), pour pouvoir les relire sans relancer le code. Le fichier ne contient que des statistiques agrégées.

In [ ]:
from utils.exploration import controle_groupes, risk_rate_by_modality, exporter_excel
from utils.segmentation import rank_segmentation_candidates, profil_groupes

defaut_a_date, _ = controle_groupes(base, {
    "arr_90j": pl.col("NBJ_ARR_SS") >= 90,
    "dfo_non_resolu": pl.col("DATE_DERNIER_DFO") >= pl.col("DATE_SAIN"),
    "date_sain_null": pl.col("DATE_SAIN").is_null(),
    "dfo_M1_M3": pl.col("Dfo_max_Mm1_Mm3") == 1,
}, TARGET)

arbres = pl.concat([
    tree_segmentation(cand, STRUCTURELLES, TARGET, max_depth=prof, min_leaf_share=0.05)[0]
    .with_columns(profondeur=pl.lit(prof))
    for prof in (2, 3)
])

exporter_excel({
    "1_Perimetre": suivi_perimetre,
    "1_Defaut_a_date": defaut_a_date,
    "1_Cible_par_mois": risk_rate_by_modality(base, DATE, TARGET).sort(DATE),
    "2_Candidats": rank_segmentation_candidates(cand, CANDIDATS, TARGET),
    "2_Arbres_CART": arbres,
    "2_AFDM_variance": resultat_afdm["variance"],
    "2_AFDM_liens_variables": resultat_afdm["liens"],
    "2_Kmeans_qualite": qualite,
    "2_Kmeans_profil_groupes": profil_groupes(cand, "cluster", TARGET, QUANTI_AFDM, QUALI_AFDM),
    "2_Volumes_decoupages": volumes_decoupages,
    "2_Comparaison_decoupages": comparaison,
    "2_Paires_type_credit": tests_par_paire("seg_type_credit"),
    "2_Paires_mono_multi": tests_par_paire("seg_mono_multi"),
    "2_Stabilite_retenue": risk_rate_over_time(cand, SEGMENTATION_RETENUE, TARGET, DATE),
    "2_Stabilite_alternative": risk_rate_over_time(cand, "seg_mono_multi", TARGET, DATE),
}, "outputs/segmentation.xlsx")
print("export : outputs/segmentation.xlsx")


## Sauvegarde de la base segmentée
On enregistre la base du périmètre final (dossiers joints unifiés) avec la colonne `segment` de la segmentation retenue. C'est le point de départ des statistiques descriptives (notebook 0b) et de la modélisation.

In [ ]:
cand = cand.with_columns(segment=pl.col(SEGMENTATION_RETENUE))

SORTIE = Path("data/processed/base_segmentee.parquet")
cand.select(*base_perim.columns, "segment").write_parquet(SORTIE)
print(SORTIE, cand.height, "lignes")
print(cand.group_by("segment").agg(n=pl.len(), n_defaut=pl.col(TARGET).sum()).sort("segment"))